# Verify split class labels

For a given DCLDE 2027 run directory, shows which `Labels` values land in the
`train` and `test` side of every `reconstructed_splits/splits_*.csv`
(`birdnet01`..`birdnet09` and `full_train`), with per-class row counts.

Point `RUN_DIR` at the run you want to check.

In [1]:
from pathlib import Path

import pandas as pd

DATA_ROOT = Path('/home/noah/HALLO_encoder_collection/data_raw/DCLDE_2027')
RUN_DIR = DATA_ROOT / '20260827_090049'  # or: sorted(DATA_ROOT.glob('20*'))[-1] for the latest
SPLITS_DIR = RUN_DIR / 'reconstructed_splits'

RUN_DIR

PosixPath('/home/noah/HALLO_encoder_collection/data_raw/DCLDE_2027/20260827_090049')

In [2]:
# only the two columns we need -- annotations.csv is ~300 MB
anno = pd.read_csv(RUN_DIR / 'annotations.csv', usecols=['uid', 'Labels'])
print(f'{len(anno):,} annotations, {anno["Labels"].nunique()} distinct labels')
anno['Labels'].value_counts()

1,180,187 annotations, 10 distinct labels


Labels
Background    972678
HW            124977
SRKW           20890
TKW            12707
UndBio         11820
AB             11623
NRKW            8266
SAR             8078
KW_und          6450
OKW             2698
Name: count, dtype: int64

## Per-split class label counts

In [3]:
records = []
for f in sorted(SPLITS_DIR.glob('splits_*.csv')):
    name = f.stem.removeprefix('splits_')
    s = pd.read_csv(f).merge(anno, on='uid', how='left')
    assert s['uid'].is_unique, f'{f.name}: duplicate uid'
    assert s['Labels'].notna().all(), f'{f.name}: uid missing from annotations.csv'
    assert set(s['fold_0'].unique()) <= {'train', 'test'}, f'{f.name}: unexpected fold_0 value'
    for (split, label), n in s.groupby(['fold_0', 'Labels']).size().items():
        records.append({'file': name, 'split': split, 'Labels': label, 'count': int(n)})

counts = pd.DataFrame(records)
counts

,file,split,Labels,count
0,birdnet01,test,AB,2406
1,birdnet01,test,Background,168987
2,birdnet01,test,HW,25071
3,birdnet01,test,KW_und,489
4,birdnet01,test,NRKW,129
...,...,...,...,...
158,full_train,train,OKW,2623
159,full_train,train,SAR,8078
160,full_train,train,SRKW,16886
161,full_train,train,TKW,11319


### TRAIN -- class row counts per split file

In [4]:
train_tab = (
    counts[counts['split'] == 'train']
    .pivot_table(index='Labels', columns='file', values='count', fill_value=0)
    .astype(int)
)
train_tab

file,birdnet01,birdnet02,birdnet04,birdnet05,birdnet06,birdnet07,birdnet08,birdnet09,full_train
Labels,,,,,,,,,
AB,4739,4739,4749,4749,4747,4752,4752,4750,9217
Background,176413,176055,176661,176303,174952,175351,174993,173642,803691
HW,61121,61121,61120,61120,61113,61120,61120,61113,99904
KW_und,1409,1412,1459,1462,1458,1459,1462,1458,5961
NRKW,69,69,69,69,69,69,69,69,8137
OKW,181,181,181,181,181,181,181,181,2623
SAR,0,0,0,0,0,0,0,0,8078
SRKW,13036,13036,13405,13405,13405,13283,13283,13283,16886
TKW,7352,7344,7352,7344,7278,7352,7344,7278,11319


### TEST -- class row counts per split file

In [5]:
test_tab = (
    counts[counts['split'] == 'test']
    .pivot_table(index='Labels', columns='file', values='count', fill_value=0)
    .astype(int)
)
test_tab

file,birdnet01,birdnet02,birdnet04,birdnet05,birdnet06,birdnet07,birdnet08,birdnet09,full_train
Labels,,,,,,,,,
AB,2406,2406,2406,2406,2406,2406,2406,2406,2406
Background,168987,168987,168987,168987,168987,168987,168987,168987,168987
HW,25071,25071,25071,25071,25071,25071,25071,25071,25071
KW_und,489,489,489,489,489,489,489,489,489
NRKW,129,129,129,129,129,129,129,129,129
OKW,75,75,75,75,75,75,75,75,75
SRKW,3909,3909,3909,3909,3909,3909,3909,3909,3909
TKW,1381,1381,1381,1381,1381,1381,1381,1381,1381
UndBio,2123,2123,2123,2123,2123,2123,2123,2123,2123


## Which labels are present in each file / split

Also flags any label that exists in the dataset but is absent from a given split.

In [6]:
all_labels = set(anno['Labels'].unique())
present = counts.groupby(['file', 'split'])['Labels'].apply(lambda s: sorted(s.unique()))

for (file, split), labels in present.items():
    missing = sorted(all_labels - set(labels))
    tag = f'  MISSING: {missing}' if missing else ''
    print(f'{file:11s} {split:5s} ({len(labels):2d}): {labels}{tag}')

birdnet01   test  ( 9): ['AB', 'Background', 'HW', 'KW_und', 'NRKW', 'OKW', 'SRKW', 'TKW', 'UndBio']  MISSING: ['SAR']
birdnet01   train ( 9): ['AB', 'Background', 'HW', 'KW_und', 'NRKW', 'OKW', 'SRKW', 'TKW', 'UndBio']  MISSING: ['SAR']
birdnet02   test  ( 9): ['AB', 'Background', 'HW', 'KW_und', 'NRKW', 'OKW', 'SRKW', 'TKW', 'UndBio']  MISSING: ['SAR']
birdnet02   train ( 9): ['AB', 'Background', 'HW', 'KW_und', 'NRKW', 'OKW', 'SRKW', 'TKW', 'UndBio']  MISSING: ['SAR']
birdnet04   test  ( 9): ['AB', 'Background', 'HW', 'KW_und', 'NRKW', 'OKW', 'SRKW', 'TKW', 'UndBio']  MISSING: ['SAR']
birdnet04   train ( 9): ['AB', 'Background', 'HW', 'KW_und', 'NRKW', 'OKW', 'SRKW', 'TKW', 'UndBio']  MISSING: ['SAR']
birdnet05   test  ( 9): ['AB', 'Background', 'HW', 'KW_und', 'NRKW', 'OKW', 'SRKW', 'TKW', 'UndBio']  MISSING: ['SAR']
birdnet05   train ( 9): ['AB', 'Background', 'HW', 'KW_und', 'NRKW', 'OKW', 'SRKW', 'TKW', 'UndBio']  MISSING: ['SAR']
birdnet06   test  ( 9): ['AB', 'Background', 'HW

## Sanity checks

Every `splits_*.csv` evaluates on the same `holdout_eval.csv` recordings, so the
`test` column should be identical across all files.

In [7]:
ref = test_tab['full_train']
for file in test_tab.columns:
    same = test_tab[file].equals(ref)
    print(f'{file:11s} test == full_train test: {same}')

birdnet01   test == full_train test: True
birdnet02   test == full_train test: True
birdnet04   test == full_train test: True
birdnet05   test == full_train test: True
birdnet06   test == full_train test: True
birdnet07   test == full_train test: True
birdnet08   test == full_train test: True
birdnet09   test == full_train test: True
full_train  test == full_train test: True
